In [35]:
"""
export_filtered_designs.py

Export only designs matching a chosen configuration from previously saved
porous-design CSV files.

Default filter in this script
-----------------------------
- flow_direction == "pore1_to_pore2"
- topology_pore1_side == "lower"
- topology_pore2_side == "lower"
- gain_percent >= 5.0

This script is written so the filter is easy to change later.
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable

import pandas as pd


# =============================================================================
# USER CONFIGURATION
# =============================================================================
INPUT_CSVS = [
    "csv_results/all_designs_above_5_percent_lift_gain.csv",
]

OUTPUT_DIR = "filtered_design_exports"
OUTPUT_FILENAME = "filtered_lower_upper_pore1_to_pore2_gain_above_7.csv"

# -------------------------------------------------------------------------
# EASY-TO-CHANGE FILTER SETTINGS
# -------------------------------------------------------------------------
FILTERS = {
    "flow_direction": "pore1_to_pore2",
    "topology_pore1_side": "lower",
    "topology_pore2_side": "upper",
    "min_gain_percent": 7.0,
}


# =============================================================================
# HELPERS
# =============================================================================
def find_existing_csvs(paths: Iterable[str | Path]) -> list[Path]:
    """Return only CSV paths that exist."""
    found: list[Path] = []
    for p in paths:
        pp = Path(p)
        if pp.exists():
            found.append(pp)
    return found


def normalize_side(value: str) -> str:
    """Normalize side labels to lower-case."""
    return str(value).strip().lower()


def flow_direction_from_q(q: float) -> str:
    """
    Compute flow direction from Q.

    Convention
    ----------
    - Q > 0  -> pore1_to_pore2
    - Q < 0  -> pore2_to_pore1
    - Q == 0 -> zero
    """
    if pd.isna(q):
        return "unknown"
    if q > 0.0:
        return "pore1_to_pore2"
    if q < 0.0:
        return "pore2_to_pore1"
    return "zero"


def load_and_merge_csvs(csv_paths: list[Path]) -> pd.DataFrame:
    """
    Load and merge all input CSV files.

    Returns
    -------
    pandas.DataFrame
        Merged table.
    """
    if not csv_paths:
        raise FileNotFoundError(
            "No input CSV files were found. "
            "Check INPUT_CSVS and working directory."
        )

    frames: list[pd.DataFrame] = []
    for path in csv_paths:
        df = pd.read_csv(path)
        df["source_csv"] = path.name
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


def ensure_required_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Ensure required columns exist. If flow_direction is missing but Q_m3_s
    exists, compute it automatically.
    """
    out = df.copy()

    required_base = {
        "topology_pore1_side",
        "topology_pore2_side",
        "gain_percent",
    }
    missing_base = required_base - set(out.columns)
    if missing_base:
        raise ValueError(f"Missing required columns: {sorted(missing_base)}")

    out["topology_pore1_side"] = out["topology_pore1_side"].map(normalize_side)
    out["topology_pore2_side"] = out["topology_pore2_side"].map(normalize_side)
    out["gain_percent"] = pd.to_numeric(out["gain_percent"], errors="coerce")

    if "flow_direction" not in out.columns:
        if "Q_m3_s" not in out.columns:
            raise ValueError(
                "Missing both 'flow_direction' and 'Q_m3_s'. "
                "Need one of them to filter by direction."
            )
        out["Q_m3_s"] = pd.to_numeric(out["Q_m3_s"], errors="coerce")
        out["flow_direction"] = out["Q_m3_s"].map(flow_direction_from_q)
    else:
        out["flow_direction"] = out["flow_direction"].astype(str).str.strip()

    return out


def apply_filters(df: pd.DataFrame, filters: dict) -> pd.DataFrame:
    """
    Apply the configured filters.

    Supported keys
    --------------
    - flow_direction
    - topology_pore1_side
    - topology_pore2_side
    - min_gain_percent
    """
    out = df.copy()

    if "flow_direction" in filters and filters["flow_direction"] is not None:
        out = out[out["flow_direction"] == filters["flow_direction"]]

    if "topology_pore1_side" in filters and filters["topology_pore1_side"] is not None:
        out = out[out["topology_pore1_side"] == normalize_side(filters["topology_pore1_side"])]

    if "topology_pore2_side" in filters and filters["topology_pore2_side"] is not None:
        out = out[out["topology_pore2_side"] == normalize_side(filters["topology_pore2_side"])]

    if "min_gain_percent" in filters and filters["min_gain_percent"] is not None:
        out = out[out["gain_percent"] >= float(filters["min_gain_percent"])]

    return out


def deduplicate_designs(df: pd.DataFrame) -> pd.DataFrame:
    """
    Deduplicate exact repeated designs across multiple source CSVs.

    Keeps the row with highest gain_percent among exact duplicates.
    """
    dedupe_cols = [
        "topology_pore1_side",
        "topology_pore2_side",
        "flow_direction",
        "x1_frac",
        "x2_frac",
        "d1_m",
        "d2_m",
        "effective_hydraulic_diameter_m",
    ]

    available_dedupe_cols = [c for c in dedupe_cols if c in df.columns]

    out = (
        df.sort_values(by="gain_percent", ascending=False)
        .drop_duplicates(subset=available_dedupe_cols, keep="first")
        .reset_index(drop=True)
    )
    return out


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    """Load, filter, deduplicate, and export the requested designs."""
    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_paths = find_existing_csvs(INPUT_CSVS)
    print("[Input] Found CSV files:")
    for p in csv_paths:
        print(f"  - {p}")

    df = load_and_merge_csvs(csv_paths)
    df = ensure_required_columns(df)
    df_filtered = apply_filters(df, FILTERS)
    df_filtered = deduplicate_designs(df_filtered)

    # Sort final export by gain descending
    df_filtered = df_filtered.sort_values(by="gain_percent", ascending=False).reset_index(drop=True)

    output_path = out_dir / OUTPUT_FILENAME
    df_filtered.to_csv(output_path, index=False)

    print()
    print("=" * 72)
    print("FILTERED EXPORT COMPLETE")
    print("=" * 72)
    print(f"Flow direction          : {FILTERS['flow_direction']}")
    print(f"Topology                : ({FILTERS['topology_pore1_side']}, {FILTERS['topology_pore2_side']})")
    print(f"Minimum gain [%]        : {FILTERS['min_gain_percent']}")
    print(f"Exported designs        : {len(df_filtered)}")
    print(f"Saved file              : {output_path}")
    print("=" * 72)


if __name__ == "__main__":
    main()

[Input] Found CSV files:
  - csv_results\all_designs_above_5_percent_lift_gain.csv

FILTERED EXPORT COMPLETE
Flow direction          : pore1_to_pore2
Topology                : (lower, upper)
Minimum gain [%]        : 7.0
Exported designs        : 0
Saved file              : filtered_design_exports\filtered_lower_upper_pore1_to_pore2_gain_above_7.csv


In [43]:
"""
merge_close_porous_networks.py

Round porous-network design variables to a chosen precision and merge designs
that are effectively the same after rounding.

What this script does
---------------------
1. Reads one or more porous-design CSV files.
2. Rounds key geometric variables to a chosen precision.
3. Groups/merges porous networks that become identical after rounding.
4. Keeps one representative design per rounded group.
5. Saves:
   - merged representative designs
   - full grouped table with group ids
   - summary table for each merged group

Important note on precision
---------------------------
You asked for rounding to "10e4". In numerical notation that usually means
10^(-4) precision in practice for rounding design variables, i.e. 0.0001.

So this script uses:
    ROUND_DECIMALS = 4

That means values are rounded to 4 decimal places.

If you want a different meaning, change only:
    ROUND_DECIMALS
"""

from __future__ import annotations

from pathlib import Path
from typing import Iterable

import pandas as pd


# =============================================================================
# USER CONFIGURATION
# =============================================================================
INPUT_CSVS = [
    r"C:\Users\kusha\OneDrive\Desktop\airfoil_rewrite\porous_airfoil_02_04_2026\filtered_design_exports/filtered_lower_lower_pore1_to_pore2_gain_above_5.csv",
    r"C:\Users\kusha\OneDrive\Desktop\airfoil_rewrite\porous_airfoil_02_04_2026\filtered_design_exports/filtered_lower_upper_pore2_to_pore1_gain_above_7.csv",
]

OUTPUT_DIR = "merged_close_porous_networks"

# Rounding precision: 4 decimals = 1e-4
ROUND_DECIMALS = 4

# Columns used to define whether two porous networks are "the same"
# after rounding.
GROUP_COLS = [
    "topology_pore1_side",
    "topology_pore2_side",
    "x1_frac",
    "x2_frac",
    "d1_m",
    "d2_m",
    "effective_hydraulic_diameter_m",
]

# If present, these columns are useful for ranking the representative row
# inside each merged group.
PREFERENCE_SORT_COLS = [
    "gain_percent",
    "balanced_score",
    "CL",
]
PREFERENCE_SORT_ASCENDING = [False, False, False]


# =============================================================================
# HELPERS
# =============================================================================
def find_existing_csvs(paths: Iterable[str | Path]) -> list[Path]:
    """Return only CSV paths that exist."""
    found: list[Path] = []
    for p in paths:
        pp = Path(p)
        if pp.exists():
            found.append(pp)
    return found


def normalize_side(value: str) -> str:
    """Normalize side labels."""
    return str(value).strip().lower()


def load_csvs(csv_paths: list[Path]) -> pd.DataFrame:
    """
    Load and concatenate all input CSV files.

    Returns
    -------
    pandas.DataFrame
        Combined table.
    """
    if not csv_paths:
        raise FileNotFoundError("No input CSV files were found.")

    frames: list[pd.DataFrame] = []
    for path in csv_paths:
        df = pd.read_csv(path)
        df["source_csv"] = path.name
        frames.append(df)

    return pd.concat(frames, ignore_index=True)


def prepare_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Normalize columns and round grouping variables.

    Returns
    -------
    pandas.DataFrame
        Prepared dataframe with added rounded columns.
    """
    out = df.copy()

    required = {
        "topology_pore1_side",
        "topology_pore2_side",
        "x1_frac",
        "x2_frac",
        "d1_m",
        "d2_m",
        "effective_hydraulic_diameter_m",
    }
    missing = required - set(out.columns)
    if missing:
        raise ValueError(f"Missing required columns: {sorted(missing)}")

    out["topology_pore1_side"] = out["topology_pore1_side"].map(normalize_side)
    out["topology_pore2_side"] = out["topology_pore2_side"].map(normalize_side)

    numeric_cols = [
        "x1_frac",
        "x2_frac",
        "d1_m",
        "d2_m",
        "effective_hydraulic_diameter_m",
    ]
    for col in numeric_cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # Add rounded columns used for grouping
    for col in numeric_cols:
        out[f"{col}_rounded"] = out[col].round(ROUND_DECIMALS)

    return out


def get_grouping_columns(df: pd.DataFrame) -> list[str]:
    """
    Return actual grouping columns, using rounded versions for numeric columns.
    """
    grouping_cols: list[str] = []
    for col in GROUP_COLS:
        rounded_col = f"{col}_rounded"
        if rounded_col in df.columns:
            grouping_cols.append(rounded_col)
        else:
            grouping_cols.append(col)
    return grouping_cols


def assign_group_ids(df: pd.DataFrame) -> pd.DataFrame:
    """
    Assign a merged group id to each row.

    Returns
    -------
    pandas.DataFrame
        Dataframe with group_id column.
    """
    out = df.copy()
    grouping_cols = get_grouping_columns(out)

    out["group_id"] = (
        out.groupby(grouping_cols, dropna=False)
        .ngroup()
        .astype(int) + 1
    )

    return out


def choose_representative_rows(df: pd.DataFrame) -> pd.DataFrame:
    """
    Choose one representative row per merged group.

    The representative is chosen by sorting using the preferred ranking columns
    if they exist. Otherwise the first row in the group is kept.

    Returns
    -------
    pandas.DataFrame
        Representative rows only.
    """
    out = df.copy()

    available_sort_cols = [c for c in PREFERENCE_SORT_COLS if c in out.columns]
    available_sort_ascending = PREFERENCE_SORT_ASCENDING[:len(available_sort_cols)]

    if available_sort_cols:
        out = out.sort_values(
            by=["group_id"] + available_sort_cols,
            ascending=[True] + available_sort_ascending,
        )
    else:
        out = out.sort_values(by=["group_id"])

    reps = out.groupby("group_id", as_index=False).head(1).copy()
    reps = reps.sort_values(by="group_id").reset_index(drop=True)
    return reps


def build_group_summary(df: pd.DataFrame) -> pd.DataFrame:
    """
    Build one summary row per merged group.

    Returns
    -------
    pandas.DataFrame
        Group summary table.
    """
    rows: list[dict] = []

    for group_id, sub in df.groupby("group_id", sort=True):
        row0 = sub.iloc[0]

        summary = {
            "group_id": int(group_id),
            "n_rows_merged": int(len(sub)),
            "topology_pore1_side": row0["topology_pore1_side"],
            "topology_pore2_side": row0["topology_pore2_side"],
            "x1_frac_rounded": row0.get("x1_frac_rounded"),
            "x2_frac_rounded": row0.get("x2_frac_rounded"),
            "d1_m_rounded": row0.get("d1_m_rounded"),
            "d2_m_rounded": row0.get("d2_m_rounded"),
            "effective_hydraulic_diameter_m_rounded": row0.get("effective_hydraulic_diameter_m_rounded"),
        }

        if "gain_percent" in sub.columns:
            summary["gain_percent_max"] = pd.to_numeric(sub["gain_percent"], errors="coerce").max()
            summary["gain_percent_mean"] = pd.to_numeric(sub["gain_percent"], errors="coerce").mean()
            summary["gain_percent_min"] = pd.to_numeric(sub["gain_percent"], errors="coerce").min()

        if "Q_m3_s" in sub.columns:
            q = pd.to_numeric(sub["Q_m3_s"], errors="coerce")
            summary["Q_m3_s_mean"] = q.mean()

        if "flow_direction" in sub.columns:
            summary["flow_direction_mode"] = (
                sub["flow_direction"].astype(str).mode().iloc[0]
                if not sub["flow_direction"].astype(str).mode().empty
                else ""
            )

        if "source_csv" in sub.columns:
            summary["source_csvs"] = "|".join(sorted(set(sub["source_csv"].astype(str))))

        rows.append(summary)

    return pd.DataFrame(rows)


def drop_helper_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Drop helper rounding columns but keep original CSV structure plus group_id.
    """
    out = df.copy()
    helper_cols = [c for c in out.columns if c.endswith("_rounded")]
    return out.drop(columns=helper_cols, errors="ignore")


# =============================================================================
# MAIN
# =============================================================================
def main() -> None:
    """Round, merge close porous networks, and export results."""
    out_dir = Path(OUTPUT_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)

    csv_paths = find_existing_csvs(INPUT_CSVS)
    print("[Input] Found CSV files:")
    for p in csv_paths:
        print(f"  - {p}")

    df = load_csvs(csv_paths)
    n_before = len(df)

    df_prepared = prepare_dataframe(df)
    df_grouped = assign_group_ids(df_prepared)

    representatives = choose_representative_rows(df_grouped)
    group_summary = build_group_summary(df_grouped)

    df_grouped_export = drop_helper_columns(df_grouped)
    representatives_export = drop_helper_columns(representatives)

    n_after = len(representatives_export)
    n_merged_out = n_before - n_after

    grouped_path = out_dir / "all_rows_with_group_ids.csv"
    reps_path = out_dir / "merged_representative_networks.csv"
    summary_path = out_dir / "merged_group_summary.csv"

    df_grouped_export.to_csv(grouped_path, index=False)
    representatives_export.to_csv(reps_path, index=False)
    group_summary.to_csv(summary_path, index=False)

    print()
    print("=" * 72)
    print("MERGE CLOSE POROUS NETWORKS COMPLETE")
    print("=" * 72)
    print(f"Rounding decimals used         : {ROUND_DECIMALS}")
    print(f"Rows before merging            : {n_before}")
    print(f"Representative rows after merge: {n_after}")
    print(f"Rows merged out                : {n_merged_out}")
    print(f"[CSV Saved] {grouped_path}")
    print(f"[CSV Saved] {reps_path}")
    print(f"[CSV Saved] {summary_path}")
    print("=" * 72)


if __name__ == "__main__":
    main()

[Input] Found CSV files:
  - C:\Users\kusha\OneDrive\Desktop\airfoil_rewrite\porous_airfoil_02_04_2026\filtered_design_exports\filtered_lower_lower_pore1_to_pore2_gain_above_5.csv
  - C:\Users\kusha\OneDrive\Desktop\airfoil_rewrite\porous_airfoil_02_04_2026\filtered_design_exports\filtered_lower_upper_pore2_to_pore1_gain_above_7.csv

MERGE CLOSE POROUS NETWORKS COMPLETE
Rounding decimals used         : 4
Rows before merging            : 670
Representative rows after merge: 587
Rows merged out                : 83
[CSV Saved] merged_close_porous_networks\all_rows_with_group_ids.csv
[CSV Saved] merged_close_porous_networks\merged_representative_networks.csv
[CSV Saved] merged_close_porous_networks\merged_group_summary.csv
